In [76]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_TokenBiLSTMModel, load_LoRASEModel, load_AdapterModel
from graph_utils import get_graph_embeddings_from_string_with_model, get_bilstm_embeddings_from_string_with_model, get_token_bilstm_embeddings_from_string_with_model, get_adapter_embeddings_from_string_with_model, make_graph_ready_for_token_ids
import torch
import numpy as np
import pickle
from GridMLM_tokenizers import CSGridMLMTokenizer
from eval_utils import ensure_in_seq_string_form, cos
import pandas as pd
import os

In [77]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [78]:
device_name = 'cuda:2'
device = torch.device(device_name)

guide_arch = 'LoRA'
contra = True

adapter_model_path = f'saved_models/{guide_arch}/adapter/adapter_model_' + contra*'contra_' + 'jnhw.pt'
graph_adapter_model_path = f'saved_models/{guide_arch}/adapter/graph_model_' + contra*'contra_' + 'jnhw.pt'
token_adapter_model_path = f'saved_models/{guide_arch}/adapter/bilstm_model_' + contra*'contra_' + 'jnhw.pt'

graph_model_path = f'saved_models/{guide_arch}/graph/graph_model_' + contra*'contra_' + 'jnhw.pt'
token_model_path = f'saved_models/{guide_arch}/token_bilstm/bilstm_model_' + contra*'contra_' + 'jnhw.pt'

token_adapter_model = load_TokenBiLSTMModel(token_adapter_model_path, tokenizer, device)
graph_adapter_model = load_GraphModel(graph_adapter_model_path, device)
adapter_model = load_AdapterModel(adapter_model_path, device)

token_model = load_TokenBiLSTMModel(token_adapter_model_path, tokenizer, device)
graph_model = load_GraphModel(graph_adapter_model_path, device)

token_adapter_model.eval()
graph_adapter_model.eval()
adapter_model.eval()

token_model.eval()
graph_model.eval()

HarmonicGraphEncoder(
  (pitch_embedding): Embedding(12, 256)
  (pitch_proj): Linear(in_features=256, out_features=256, bias=True)
  (event_proj): Linear(in_features=1, out_features=256, bias=True)
  (participation_mpnn): ParticipationMPNN()
  (temporal_mpnn): TemporalMPNN()
  (to_latent): Linear(in_features=256, out_features=512, bias=True)
)

In [79]:
test_transitions = []

test_transitions.append({
    'type': 'types',
    'basic': ['G:7', 'C:maj'],
    'similar': ['G:maj', 'C:maj7'],
    'odd': ['G:min', 'C:min']
})
# s1 = ['G:7', 'C:maj']
# s2 = ['G:maj', 'C:maj7']
# s3 = ['G:7', 'C:maj7']

test_transitions.append({
    'type': 'root',
    'basic': ['G:7', 'C:maj'],
    'similar': ['G:7', 'A:min7'],
    'odd': ['E:7', 'A:maj']
})
# s1 = ['G:7', 'C:maj']
# s2 = ['G:maj', 'A:min7']
# s3 = ['E:7', 'A:min7']

test_transitions.append({
    'type': 'number',
    'basic': ['A:min', 'D:min7', 'G:7', 'C:maj'],
    'similar': ['A:min', 'G:7', 'C:maj'],
    'odd': ['A:min', 'D:min7', 'G:7']
})
# s1 = ['A:7', 'C:7']
# s2 = ['A:7', 'D:7', 'C:7']
# s3 = ['A:7', 'D:7', 'G:7', 'C:7']

test_transitions.append({
    'type': 'order',
    'basic': ['A:min', 'D:min7', 'G:7', 'C:maj'],
    'similar': ['D:min7', 'A:min', 'G:7', 'C:maj'],
    'odd': ['C:maj', 'G:7', 'D:min7', 'A:min']
})

# s1 = ['G:7', 'C:7']
# s2 = ['A:7', 'D:7']
# s3 = ['F:7', 'A#:7']

test_transitions.append({
    'type': 'new transition',
    'basic': ['A:min', 'C#:maj'],
    'similar': ['A:min7', 'C#:maj6'],
    'odd': ['A#:min', 'D:maj']
})
# 'b_A:min_@2;C#:maj_@2',
# s1 = ['A:min', 'C#:maj']
# s2 = ['A:min7', 'C#:maj6']
# s3 = ['A#:min', 'D:maj']

# # 'b_G:maj_@2;A#:11_@2',
# s1 = ['G:maj', 'A#:11']
# s2 = ['G:maj6', 'A#:7']
# s3 = ['G#:maj', 'B:11']

test_transitions.append({
    'type': 'new chords',
    'basic': ['E:maj13', 'G#:sus2'],
    'similar': ['E:maj', 'G#:9'],
    'odd': ['F:maj', 'A:9']
})

# s1 = ['E:maj13', 'G#:sus2']
# s2 = ['E:maj', 'G#:9']
# s3 = ['F:maj', 'A:9']

# s1 = ['D#:minmaj7', 'B:maj6']
# s2 = ['D#:min', 'B:maj']
# s3 = ['D:min', 'C:maj']

In [80]:
for t in test_transitions:
    basic = ensure_in_seq_string_form(t['basic'])
    similar = ensure_in_seq_string_form(t['similar'])
    odd = ensure_in_seq_string_form(t['odd'])

    y_graph_a_basic = get_graph_embeddings_from_string_with_model(basic, graph_adapter_model)
    y_graph_a_similar = get_graph_embeddings_from_string_with_model(similar, graph_adapter_model)
    y_graph_a_odd = get_graph_embeddings_from_string_with_model(odd, graph_adapter_model)

    y_token_a_basic = get_token_bilstm_embeddings_from_string_with_model(basic, token_adapter_model)
    y_token_a_similar = get_token_bilstm_embeddings_from_string_with_model(similar, token_adapter_model)
    y_token_a_odd = get_token_bilstm_embeddings_from_string_with_model(odd, token_adapter_model)

    y_adapter_basic = get_adapter_embeddings_from_string_with_model(basic, adapter_model, graph_adapter_model, token_adapter_model)
    y_adapter_similar = get_adapter_embeddings_from_string_with_model(similar, adapter_model, graph_adapter_model, token_adapter_model)
    y_adapter_odd = get_adapter_embeddings_from_string_with_model(odd, adapter_model, graph_adapter_model, token_adapter_model)

    y_graph_basic = get_graph_embeddings_from_string_with_model(basic, graph_model)
    y_graph_similar = get_graph_embeddings_from_string_with_model(similar, graph_model)
    y_graph_odd = get_graph_embeddings_from_string_with_model(odd, graph_model)

    y_token_basic = get_token_bilstm_embeddings_from_string_with_model(basic, token_model)
    y_token_similar = get_token_bilstm_embeddings_from_string_with_model(similar, token_model)
    y_token_odd = get_token_bilstm_embeddings_from_string_with_model(odd, token_model)

    t['similar-token'] = cos(y_token_basic, y_token_similar).item()
    t['odd-token'] = cos(y_token_basic, y_token_odd).item()

    t['similar-graph'] = cos(y_graph_basic, y_graph_similar).item()
    t['odd-graph'] = cos(y_graph_basic, y_graph_odd).item()

    t['similar-adapter'] = cos(y_adapter_basic, y_adapter_similar).item()
    t['odd-adapter'] = cos(y_adapter_basic, y_adapter_odd).item()

    t['similar-token-a'] = cos(y_token_a_basic, y_token_a_similar).item()
    t['odd-token-a'] = cos(y_token_a_basic, y_token_a_odd).item()

    t['similar-graph-a'] = cos(y_graph_a_basic, y_graph_a_similar).item()
    t['odd-graph-a'] = cos(y_graph_a_basic, y_graph_a_odd).item()

In [81]:
for t in test_transitions:
    print(t)

{'type': 'types', 'basic': ['G:7', 'C:maj'], 'similar': ['G:maj', 'C:maj7'], 'odd': ['G:min', 'C:min'], 'similar-token': 0.5096967220306396, 'odd-token': -0.028570733964443207, 'similar-graph': 0.6258608102798462, 'odd-graph': 0.34209734201431274, 'similar-adapter': 0.6208630204200745, 'odd-adapter': 0.3297472298145294, 'similar-token-a': 0.5096967220306396, 'odd-token-a': -0.028570733964443207, 'similar-graph-a': 0.6258608102798462, 'odd-graph-a': 0.34209734201431274}
{'type': 'root', 'basic': ['G:7', 'C:maj'], 'similar': ['G:7', 'A:min7'], 'odd': ['E:7', 'A:maj'], 'similar-token': 1.0, 'odd-token': 0.05736732482910156, 'similar-graph': 1.0, 'odd-graph': 0.3355342745780945, 'similar-adapter': 1.0, 'odd-adapter': 0.27998119592666626, 'similar-token-a': 1.0, 'odd-token-a': 0.05736732482910156, 'similar-graph-a': 1.0, 'odd-graph-a': 0.3355342745780945}
{'type': 'number', 'basic': ['A:min', 'D:min7', 'G:7', 'C:maj'], 'similar': ['A:min', 'G:7', 'C:maj'], 'odd': ['A:min', 'D:min7', 'G:7'],

In [90]:
df = pd.DataFrame(test_transitions)
# # latex-safe - replace '#' with '\\#'
# df = df.apply(lambda col: col.map(
#     lambda x: x.replace("#", r"\#") if isinstance(x, str) else x
# ))
# df.style.format(precision=2)

In [ ]:
def latex_safe(value):
    if isinstance(value, str):
        return value.replace("#", r"\#")
    if isinstance(value, (list, tuple, set, dict)):
        return str(value).replace("#", r"\#")
    return value

df_latex = df.copy()
df_latex = df_latex.applymap(latex_safe)

latex_tables_base_path = 'results/latex_tables/'
os.makedirs(latex_tables_base_path, exist_ok=True)

df_latex.to_latex(
    latex_tables_base_path + 'similar_odd_transitions.tex',
    index=False,
    float_format="%.2f"
)

/tmp/ipykernel_1794280/2444643456.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_latex = df_latex.applymap(latex_safe)


In [83]:
def latex_safe(obj):
    if isinstance(obj, str):
        return obj.replace("#", r"\#")
    if isinstance(obj, list):
        return [latex_safe(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(latex_safe(x) for x in obj)
    if isinstance(obj, dict):
        return {k: latex_safe(v) for k, v in obj.items()}
    return obj

df = df.apply(lambda col: col.map(latex_safe))

In [84]:
print(df)

             type                        basic                      similar  \
0           types                 [G:7, C:maj]              [G:maj, C:maj7]   
1            root                 [G:7, C:maj]                [G:7, A:min7]   
2          number  [A:min, D:min7, G:7, C:maj]          [A:min, G:7, C:maj]   
3           order  [A:min, D:min7, G:7, C:maj]  [D:min7, A:min, G:7, C:maj]   
4  new transition             [A:min, C\#:maj]           [A:min7, C\#:maj6]   
5      new chords          [E:maj13, G\#:sus2]               [E:maj, G\#:9]   

                           odd  similar-token  odd-token  similar-graph  \
0               [G:min, C:min]       0.509697  -0.028571       0.625861   
1                 [E:7, A:maj]       1.000000   0.057367       1.000000   
2         [A:min, D:min7, G:7]       0.538805   1.000000       0.760298   
3  [C:maj, G:7, D:min7, A:min]       0.355546   0.155802       0.707061   
4             [A\#:min, D:maj]       0.607390  -0.073625       0.558393

In [89]:
latex_tables_base_path = 'results/latex_tables/'
os.makedirs(latex_tables_base_path, exist_ok=True)
df.style.format(precision=2).to_latex(latex_tables_base_path + 'similar_odd_transitions.tex')

In [86]:
# y_graph_s1 = get_graph_embeddings_from_string_with_model(st1, graph_adapter_model)
# y_graph_s2 = get_graph_embeddings_from_string_with_model(st2, graph_adapter_model)
# y_graph_s3 = get_graph_embeddings_from_string_with_model(st3, graph_adapter_model)

# y_token_s1 = get_token_bilstm_embeddings_from_string_with_model(st1, token_adapter_model)
# y_token_s2 = get_token_bilstm_embeddings_from_string_with_model(st2, token_adapter_model)
# y_token_s3 = get_token_bilstm_embeddings_from_string_with_model(st3, token_adapter_model)

# y_adapter_s1 = get_adapter_embeddings_from_string_with_model(st1, adapter_model, graph_adapter_model, token_adapter_model)
# y_adapter_s2 = get_adapter_embeddings_from_string_with_model(st2, adapter_model, graph_adapter_model, token_adapter_model)
# y_adapter_s3 = get_adapter_embeddings_from_string_with_model(st3, adapter_model, graph_adapter_model, token_adapter_model)

In [87]:
# print(f's1: {s1} | s2: {s2} | s3: {s3}')

# print('graph st1-st2: ', cos(y_graph_s1, y_graph_s2))
# print('graph st2-st3: ', cos(y_graph_s2, y_graph_s3))
# print('graph st1-st3: ', cos(y_graph_s1, y_graph_s3))

# print('token st1-st2: ', cos(y_token_s1, y_token_s2))
# print('token st2-st3: ', cos(y_token_s2, y_token_s3))
# print('token st1-st3: ', cos(y_token_s1, y_token_s3))

# print('adapter st1-st2: ', cos(y_adapter_s1, y_adapter_s2))
# print('adapter st2-st3: ', cos(y_adapter_s2, y_adapter_s3))
# print('adapter st1-st3: ', cos(y_adapter_s1, y_adapter_s3))